# ITSMe DBH with stem-lean correction (step 1)

New notebook (23 Oct 2026). Based on `ITSMe_HOME_PORTABLE.ipynb`; the original course notebook and Codex's copies are unchanged.

**Problem.** `ITSMe::dbh_pc` always takes a *horizontal* slice 1.3 m above the ground. On a leaning stem that slice is an ellipse, so the circle fit over-estimates DBH (trees 103: 17 deg, 108: 26 deg). On some stems the 1.3 m slice is too sparse to fit (tree 102).

**What this notebook does, per tree**
1. Loads the same prepared local-coordinate XYZ file and foot-ground point (ground at Z = 0) used by the earlier runs.
2. Takes the stem axis Codex fitted for the height calculation (`checkpoints/tree_NNN_height.json`: robust circle centres on 0.25 m sections over the lowest ~4 m). If that file is missing, it tracks the axis itself.
3. Rotates the tree about the axis point at foot-ground level so the stem axis is vertical (temporary, in memory only; files and coordinates are not changed).
4. Estimates the bark radius as the most common point distance from the axis at 1.0-1.6 m, and keeps only points within 10 cm outside it. This removes understorey, basal flare beyond the bark line and neighbouring stems from the slice.
5. Runs the unchanged `ITSMe::dbh_pc` at 1.3 m along the stem, and `ITSMe::diameter_slice_pc` at 1.1-1.5 m.
6. Accepts a fit only if arc coverage >= 0.6 and circle RMSE <= 10% of the diameter (min 1 cm, max 4 cm). Corrected DBH = the 1.3 m fit if accepted, otherwise the median of accepted fits at 1.1-1.5 m.

Upright trees act as a control: their corrected DBH should be close to the earlier horizontal-slice value.

In [ ]:
# Settings
TREES <- NULL            # NULL = all ten selected trees; or e.g. c("tree_103", "tree_108")
AXIS_HEIGHTS <- seq(0.4, 2.6, by = 0.1)   # m above foot ground used to track the stem centre
SEARCH_RADIUS <- 0.6     # m, how far the centre may move between successive 10 cm steps
SLICE_HEIGHTS <- c(1.1, 1.2, 1.3, 1.4, 1.5)
MIN_ARC <- 0.6           # minimum ITSMe arc coverage to accept a fit
MAX_RMSE_M <- 0.04       # maximum circle-fit RMSE (m) to accept a fit
set.seed(6003)
options(warn = 1)

In [ ]:
find_root <- function() {
  configured <- Sys.getenv("INFO6003_PROJECT_ROOT", "")
  p <- normalizePath(if (nzchar(configured)) configured else getwd(), winslash = "/", mustWork = TRUE)
  repeat {
    if (dir.exists(file.path(p, "manual_seg_v2")) && dir.exists(file.path(p, "measurement_pipeline"))) return(p)
    parent <- dirname(p)
    if (identical(parent, p)) stop("Open Jupyter inside Codex_help or set INFO6003_PROJECT_ROOT.")
    p <- parent
  }
}
ROOT <- find_root(); WORK <- file.path(ROOT, "measurement_pipeline")
suppressPackageStartupMessages({library(ITSMe); library(ggplot2)})
manifest <- read.csv(file.path(WORK, "environment", "home_selection_manifest.csv"), check.names = FALSE,
                     stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
if (is.null(TREES)) TREES <- manifest$stable_tree_id
OUT <- file.path(WORK, "results", paste0("ITSMe_DBH_lean_correction_", format(Sys.time(), "%Y%m%d_%H%M%S")))
if (dir.exists(OUT)) stop("Output folder exists; rerun later.")
dir.create(OUT, recursive = TRUE)
writeLines(c(capture.output(sessionInfo()), paste("ITSMe", packageVersion("ITSMe"))), file.path(OUT, "session_info.txt"))
ground0 <- data.frame(X = 0, Y = 0, Z = 0)   # foot-ground point in the local frame
cat("Trees:", TREES, "\nOutput:", OUT, "\n")

In [ ]:
# Helpers: robust circle fit and stem-axis tracking (selection aids only; DBH itself comes from ITSMe)
kasa <- function(x, y) {
  p <- unname(qr.solve(cbind(x, y, 1), x^2 + y^2))
  cx <- p[1] / 2; cy <- p[2] / 2
  c(cx, cy, sqrt(p[3] + cx^2 + cy^2))
}
robust_circle <- function(x, y) {
  if (length(x) < 8) return(NULL)
  f <- tryCatch(kasa(x, y), error = function(e) NULL)
  for (i in 1:3) {
    if (is.null(f)) return(NULL)
    res <- abs(sqrt((x - f[1])^2 + (y - f[2])^2) - f[3])
    keep <- res <= max(0.02, 3 * median(res))
    if (sum(keep) < 8) break
    x <- x[keep]; y <- y[keep]
    f <- tryCatch(kasa(x, y), error = function(e) NULL)
  }
  if (is.null(f)) NULL else c(f, length(x))
}
track_axis <- function(pc) {
  prev <- c(0, 0); out <- NULL
  for (h in AXIS_HEIGHTS) {
    s <- pc[abs(pc$Z - h) <= 0.05, ]
    s <- s[sqrt((s$X - prev[1])^2 + (s$Y - prev[2])^2) <= SEARCH_RADIUS, ]
    f <- robust_circle(s$X, s$Y)
    if (is.null(f) || f[3] < 0.02 || f[3] > 0.7) next
    out <- rbind(out, c(x = f[[1]], y = f[[2]], z = h, r = f[[3]], n = f[[4]]))
    prev <- f[1:2]
  }
  out
}
rotation_to_vertical <- function(v) {   # Rodrigues rotation taking unit vector v onto +Z
  v <- v / sqrt(sum(v^2)); z <- c(0, 0, 1)
  k <- c(v[2] * z[3] - v[3] * z[2], v[3] * z[1] - v[1] * z[3], v[1] * z[2] - v[2] * z[1])
  s <- sqrt(sum(k^2)); c0 <- sum(v * z)
  if (s < 1e-12) return(diag(3))
  K <- matrix(c(0, -k[3], k[2], k[3], 0, -k[1], -k[2], k[1], 0), 3, byrow = TRUE)
  diag(3) + K + K %*% K * ((1 - c0) / s^2)
}
# RMSE limit is relative for thin stems: at most 10% of the diameter (min 1 cm), never above MAX_RMSE_M
accept <- function(arc, R2, d) is.finite(d) && d > 0.03 && d < 2 && arc >= MIN_ARC &&
  sqrt(R2) <= min(MAX_RMSE_M, max(0.01, 0.1 * d))

In [ ]:
# Main loop
results <- list(); slices <- list()
for (tid in TREES) {
  sel <- manifest[manifest$stable_tree_id == tid, , drop = FALSE]
  stopifnot(nrow(sel) == 1)
  pc <- ITSMe::read_tree_pc(path = file.path(ROOT, sel$local_xyz_path), samplefactor = 1)
  stopifnot(nrow(pc) == sel$point_count)
  row <- list(tree = tid)
  # Stem axis: Codex's robust low-stem axis from the height checkpoint (fitted on 0.25 m sections over ~4 m),
  # converted to the local frame. Fallback: track the axis here.
  ck <- file.path(WORK, "checkpoints", paste0(tid, "_height.json"))
  t3 <- c(sel$translation_x_m, sel$translation_y_m, sel$translation_z_m)
  if (file.exists(ck)) {
    sa <- jsonlite::fromJSON(ck)$stem_axis
    c0 <- sa$coeff[1, ]; sl <- sa$coeff[2, ]
    o <- c(c0[1] - t3[1] + sl[1] * (t3[3] - sa$z_origin), c0[2] - t3[2] + sl[2] * (t3[3] - sa$z_origin), 0)
    v <- c(sl[1], sl[2], 1); row$axis_source <- "Codex height checkpoint"
  } else {
    ax <- track_axis(pc)
    if (is.null(ax) || nrow(ax) < 5) { row$status <- "AXIS_NOT_FOUND"; results[[tid]] <- row; next }
    bx <- unname(coef(lm(x ~ z, data = as.data.frame(ax)))); by <- unname(coef(lm(y ~ z, data = as.data.frame(ax))))
    o <- c(bx[1], by[1], 0); v <- c(bx[2], by[2], 1); row$axis_source <- "tracked here"
  }
  v <- v / sqrt(sum(v^2))
  row$axis_lean_deg <- acos(v[3]) * 180 / pi
  row$axis_offset_at_foot_m <- sqrt(o[1]^2 + o[2]^2)
  # rotate about the axis point at foot level, so the foot ground stays at Z = 0
  Rm <- rotation_to_vertical(v)
  P <- sweep(as.matrix(pc[, c("X", "Y", "Z")]), 2, o) %*% t(Rm)
  rp <- data.frame(X = P[, 1], Y = P[, 2], Z = P[, 3])
  # bark radius = most common distance from the axis at 1.0-1.6 m (stem surface), then crop 10 cm outside it
  b <- rp[rp$Z >= 1.0 & rp$Z <= 1.6, ]; rho <- sqrt(b$X^2 + b$Y^2); rho <- rho[rho <= 0.8]
  if (length(rho) < 10) { row$status <- "TOO_FEW_POINTS_NEAR_AXIS"; results[[tid]] <- row; next }
  hcount <- tabulate(pmax(1, ceiling(rho / 0.01)), nbins = 80)
  hs <- stats::filter(hcount, rep(1 / 3, 3), sides = 2); hs[is.na(hs)] <- 0
  row$bark_radius_est_m <- (which.max(hs) - 0.5) * 0.01
  crop_r <- max(0.15, row$bark_radius_est_m + 0.10)
  rc <- rp[sqrt(rp$X^2 + rp$Y^2) <= crop_r, ]
  row$crop_radius_m <- crop_r
  d13 <- tryCatch(ITSMe::dbh_pc(pc = rc, dtm = ground0, r = 1.5, thresholdR2 = 0.001, slice_thickness = 0.06,
                                functional = TRUE, concavity = 4, plot = TRUE), error = function(e) NULL)
  if (!is.null(d13)) {
    row$dbh13_m <- d13$dbh; row$fdbh13_m <- d13$fdbh; row$dbh13_arc <- d13$arc_coverage; row$dbh13_R2 <- d13$R2
    row$dbh13_points <- sum(abs(rc$Z - 1.3) <= 0.03)
    ggsave(file.path(OUT, paste0(tid, "_dbh_lean_corrected.png")), plot = d13$plot, width = 7, height = 7)
  }
  acc <- c()
  for (h in SLICE_HEIGHTS) {
    s <- tryCatch(ITSMe::diameter_slice_pc(pc = rc, slice_height = h, slice_thickness = 0.06, dtm = ground0, r = 1.5,
                                           functional = TRUE, plot = FALSE), error = function(e) NULL)
    ok <- !is.null(s) && accept(s$arc_coverage, s$R2, s$diameter)
    slices[[length(slices) + 1]] <- data.frame(tree = tid, height_m = h,
      diameter_m = if (is.null(s)) NA else s$diameter, fdiameter_m = if (is.null(s)) NA else s$fdiameter,
      arc = if (is.null(s)) NA else s$arc_coverage, rmse_m = if (is.null(s)) NA else sqrt(s$R2),
      points = sum(abs(rc$Z - h) <= 0.03), accepted = ok)
    if (ok) acc <- c(acc, s$diameter)
  }
  row$accepted_slices <- length(acc)
  if (!is.null(d13) && accept(d13$arc_coverage, d13$R2, d13$dbh)) {
    row$dbh_corrected_m <- d13$dbh; row$fdbh_corrected_m <- d13$fdbh; row$method <- "1.3 m along stem"
  } else if (length(acc)) {
    row$dbh_corrected_m <- median(acc); row$fdbh_corrected_m <- NA
    row$method <- paste0("median of ", length(acc), " accepted slices, 1.1-1.5 m")
  } else {
    row$dbh_corrected_m <- NA; row$fdbh_corrected_m <- NA; row$method <- "no acceptable slice"
  }
  if (length(acc) > 1) row$slice_range_m <- paste(round(range(acc), 3), collapse = "-")
  row$status <- "OK"
  # side views before/after rotation, for checking the axis
  side <- rbind(data.frame(pc[pc$Z < 4, c("X", "Z")], view = "original (X-Z)"),
                data.frame(X = rp$X, Z = rp$Z, view = "rotated, stem vertical (X-Z)")[rp$Z < 4, ])
  g <- ggplot(side, aes(X, Z)) + geom_point(size = 0.05, alpha = 0.3) + facet_wrap(~view) + coord_fixed() +
    geom_hline(yintercept = 1.3, colour = "red") + labs(title = paste(tid, "- lower 4 m, red line = 1.3 m"))
  ggsave(file.path(OUT, paste0(tid, "_axis_check.png")), plot = g, width = 9, height = 6)
  results[[tid]] <- row
  cat(sprintf("%s lean %.1f deg, corrected DBH %s cm (%s)\n", tid, row$axis_lean_deg,
              if (is.na(row$dbh_corrected_m)) "NA" else sprintf("%.2f", 100 * row$dbh_corrected_m), row$method))
}

In [ ]:
res <- data.table::rbindlist(lapply(results, as.list), fill = TRUE)
data.table::fwrite(res, file.path(OUT, "dbh_lean_corrected.csv"))
data.table::fwrite(do.call(rbind, slices), file.path(OUT, "dbh_lean_corrected_slices.csv"))
res